In [378]:
import pandas as pd
import numpy as np
import plotly.express as px
import duckdb
import os

Inspecting the Open Food Facts dataset using DuckDB

In [379]:
file_path = "../data/raw/en.openfoodfacts.org.products.csv.gz"

con = duckdb.connect()

columns = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto(
        '{file_path}',
        delim = '\t',
        header = true,
        sample_size = 10000
    )
""").df()

columns

,column_name,column_type,null,key,default,extra
0,code,VARCHAR,YES,None,None,None
1,url,VARCHAR,YES,None,None,None
2,creator,VARCHAR,YES,None,None,None
3,created_t,BIGINT,YES,None,None,None
4,created_datetime,TIMESTAMP WITH TIME ZONE,YES,None,None,None
...,...,...,...,...,...,...
206,sulphate_100g,VARCHAR,YES,None,None,None
207,nitrate_100g,VARCHAR,YES,None,None,None
208,acidity_100g,VARCHAR,YES,None,None,None
209,carbohydrates-total_100g,DOUBLE,YES,None,None,None


In [380]:
# Columns required for the Sugar Trap project

required_columns = [
    "product_name",
    "categories_tags",
    "sugars_100g",
    "proteins_100g",
    "ingredients_text",
    "nutriscore_grade"
]

# Check which columns exist
available_columns = columns["column_name"].tolist()

for col in required_columns:
    if col in available_columns:
        print(f"FOUND: {col}")
    else:
        print(f"MISSING: {col}")

FOUND: product_name
FOUND: categories_tags
FOUND: sugars_100g
FOUND: proteins_100g
FOUND: ingredients_text
FOUND: nutriscore_grade


In [381]:
#Selected fields to carry forward with the analysis
selected_columns = [
    "product_name",
    "categories_tags",
    "sugars_100g",
    "proteins_100g",
    "ingredients_text",
    "nutriscore_grade"
]

In [382]:
# Extract selected columns from the original dataset
# Save up to 500,000 records into the processed folder

# Define the output location
output_path = "../data/processed/openfoodfacts_sample.csv"

# Construct the SQL query
query = f"""
COPY (
    SELECT {", ".join(selected_columns)}
    FROM read_csv_auto(
        '{file_path}',
        delim = '\t',
        header = true,
        sample_size = 10000         
    )
    LIMIT 500000
)
TO '{output_path}'
(HEADER, DELIMITER ',');
"""

# Execute the extraction
con.execute(query)

print("Data extraction completed successfully!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data extraction completed successfully!


In [383]:
#Confirm that the output file exists and checks its size.
print("File exists:", os.path.exists(output_path))

File exists: True


In [384]:
#Load the extracted data into a dataframe for further analysis
df =pd.read_csv(output_path)
#Display the first 10 rows of the dataframe
df.head(10)

,product_name,categories_tags,sugars_100g,proteins_100g,ingredients_text,nutriscore_grade
0,Véritable pâte à tartiner noisettes chocolat noir,"en:breakfasts,en:spreads,en:sweet-spreads,fr:p...",32.0,8.0,"HAZELNUTS, 73% cocoa dark chocolate (cocoa mas...",e
1,Chamomile Herbal Tea,en:null,NaN,NaN,CHAMOMILE FLOWERS.,unknown
2,"Lagg's, herbal tea, peppermint","en:plant-based-foods-and-beverages,en:beverage...",NaN,0.0,Peppermint.,unknown
3,Linden Flowers Tea,"en:beverages-and-beverages-preparations,en:pla...",NaN,NaN,LINDEN FLOWERS.,unknown
4,"Herbal Tea, Hibiscus",NaN,NaN,NaN,Hibiscus flowers.,unknown
5,Apple & Cinnamon Tea,en:null,NaN,NaN,"TEA, CINNAMON & NATURAL APPLE FLAVOR.",unknown
6,Green Tea,en:null,NaN,0.0,GREEN TEA.,unknown
7,Shave Grass Herbal Tea,en:null,NaN,0.0,SHAVE GRASS.,unknown
8,"Lagg's, herbal tea, chamomile * mint","en:plant-based-foods-and-beverages,en:beverage...",NaN,0.0,Chamomile spearmint.,unknown
9,Artichoke Herbal Tea,en:null,NaN,0.0,ARTICHOKE MALVA SENNA LEAF HIBISCUS CHAMOMILE ...,unknown


# Story 1: Data Cleaning

The first step of this project was to understand the dataset and assess its quality before carrying out the analysis.

I examined the dataset's structure, checked for missing values and identified nutritional measurements that fell outside the expected range. This helped me determine what needed to be cleaned before proceeding with the analysis.

In [385]:
#Check the number of rows and columns
print(df.shape)

#Check the column names
print(df.columns)

#Check data types and missing values
df.info()


(500000, 6)
Index(['product_name', 'categories_tags', 'sugars_100g', 'proteins_100g',
       'ingredients_text', 'nutriscore_grade'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   product_name      493096 non-null  object 
 1   categories_tags   402357 non-null  object 
 2   sugars_100g       453070 non-null  float64
 3   proteins_100g     462881 non-null  float64
 4   ingredients_text  369714 non-null  object 
 5   nutriscore_grade  499988 non-null  object 
dtypes: float64(2), object(4)
memory usage: 22.9+ MB


In [386]:
#Display the first 5 products
df.head(5)

,product_name,categories_tags,sugars_100g,proteins_100g,ingredients_text,nutriscore_grade
0,Véritable pâte à tartiner noisettes chocolat noir,"en:breakfasts,en:spreads,en:sweet-spreads,fr:p...",32.0,8.0,"HAZELNUTS, 73% cocoa dark chocolate (cocoa mas...",e
1,Chamomile Herbal Tea,en:null,NaN,NaN,CHAMOMILE FLOWERS.,unknown
2,"Lagg's, herbal tea, peppermint","en:plant-based-foods-and-beverages,en:beverage...",NaN,0.0,Peppermint.,unknown
3,Linden Flowers Tea,"en:beverages-and-beverages-preparations,en:pla...",NaN,NaN,LINDEN FLOWERS.,unknown
4,"Herbal Tea, Hibiscus",NaN,NaN,NaN,Hibiscus flowers.,unknown


In [387]:
#Lets count missing values in each column
missing_values =pd.isnull(df).sum()
print(missing_values)

product_name          6904
categories_tags      97643
sugars_100g          46930
proteins_100g        37119
ingredients_text    130286
nutriscore_grade        12
dtype: int64


In [388]:
#Inspect for impossible nutritional values\

# Check for negative sugar values
print("Negative sugar values:", (df["sugars_100g"] < 0).sum())

# Check for sugar values above 100
print("Sugar values above 100:", (df["sugars_100g"] > 100).sum())

# Check for negative protein values
print("Negative protein values:", (df["proteins_100g"] < 0).sum())

# Check for protein values above 100
print("Protein values above 100:", (df["proteins_100g"] > 100).sum())

Negative sugar values: 0
Sugar values above 100: 5125
Negative protein values: 1
Protein values above 100: 882


In [389]:
# Display products with impossible sugar values
sugar_errors = df[df["sugars_100g"] > 100]

sugar_errors.head(10)

,product_name,categories_tags,sugars_100g,proteins_100g,ingredients_text,nutriscore_grade
44,Knusperflakes Mit Knusprigen Cornflakes,en:null,198.0,32.0,"SUGAR, COCOA BUTTER, CREAM POWDER (16%), CORN ...",e
46,Milk Chocolate With Caramelized Almonds,en:null,224.0,28.8,"SUGAR, COCOA BUTTER, COCOA MASS, WHOLE MILK PO...",e
48,Oatmeal Raisin Cookies,en:null,102.0,25.5,"ONLY THE BEST: UN-BLEACHED FLOUR, ORGANIC STEE...",e
49,The Best Peanut Butter Cookies,en:null,102.0,25.5,"ONLY THE BEST: UN-BLEACHED FLOUR, 100% ORGANIC...",e
67,Milk Chocolate Rose,en:null,272.0,45.3,"MILK CHOCOLATE (SUGAR, MILK, COCOA BUTTER, CHO...",e
68,Solid Dark Chocolate,en:null,125.0,12.5,"SEMI-SWEET CHOCOLATE: (SUGAR, CHOCOLATE LIQUOR...",e
69,White Chocolate,en:null,125.0,18.8,"WHITE CHOCOLATE: (SUGAR, COCOA BUTTER, MILK, V...",e
75,Solid Milk Chocolate,en:null,154.0,23.1,"MILK CHOCOLATE (SUGAR, MILK, COCOA BUTTER, CHO...",e
79,Solid Dark Chocolate,en:null,132.0,13.8,"SEMI-SWEET CHOCOLATE: (SUGAR, CHOCOLATE PROCES...",e
80,Milk Chocolate Malt Balls,en:null,150.0,12.5,"MILK CHOCOLATE (SUGAR, MILK, COCOA BUTTER, CHO...",e


In [390]:
# Display products with impossible protein values
protein_errors = df[df["proteins_100g"] > 100]

protein_errors.head(10)

,product_name,categories_tags,sugars_100g,proteins_100g,ingredients_text,nutriscore_grade
496,All Natural Chocolate Raisins,en:null,153.0,119.0,"MILK CHOCOLATE (SUGAR, COCOA BUTTER, WHOLE MIL...",e
1377,Cooked Dried Pork Product,en:null,75.6,139.0,"PORK, SUGAR, SOY FLOUR, SOY SAUCE (WATER, SOYB...",e
1479,Cocoa Roast Almonds,en:null,32.6,131.0,"ALMONDS, NATURAL AND ARTIFICIAL FLAVORS, COCOA...",e
1676,Applewood Double Smoked Bacon,en:null,NaN,278.0,"CURED WITH: WATER, SALT, NATURAL APPLEWOOD SMO...",unknown
1677,Hickory Smoked Bacon,en:null,0.0,278.0,"CURED WITH: WATER, SALT, SUGAR, SODIUM PHOSPHA...",e
1973,Roasted & Salted Sunflower Kernels,en:null,0.0,312.0,"SUNFLOWER KERNELS, COTTONSEED AND/OR SUNFLOWER...",e
1974,Salad Toppers,en:null,0.0,312.0,ALMONDS,a
1975,Pine Nuts,en:null,0.0,123.0,PINE NUTS.,unknown
2090,Mozzarella Sliced Cheese,en:null,0.0,124.0,"PASTEURIZED PART-SKIM MILK, CHEESE CULTURES, S...",e
2139,Cheese & Garlic Croutons,en:null,0.0,204.0,"ENRICHED FLOUR (WHEAT FLOUR, BARLEY MALT, NIAC...",e


In [391]:
# Create a copy of the original dataset
df_clean = df.copy()

In [392]:
# Replace negative sugar values with NaN
df_clean.loc[df_clean["sugars_100g"] < 0, "sugars_100g"] = float("nan")

# Replace sugar values above 100 with NaN
df_clean.loc[df_clean["sugars_100g"] > 100, "sugars_100g"] = float("nan")

# Replace protein values above 100 with NaN
df_clean.loc[df_clean["proteins_100g"] > 100, "proteins_100g"] = float("nan")

# Replace negative protein values with NaN
df_clean.loc[df_clean["proteins_100g"] < 0, "proteins_100g"] = float("nan")

In [393]:
# Check for remaining impossible values
print("Sugar values above 100:", (df_clean["sugars_100g"] > 100).sum())

print("Protein values above 100:", (df_clean["proteins_100g"] > 100).sum())

print("Negative protein values:", (df_clean["proteins_100g"] < 0).sum())

Sugar values above 100: 0
Protein values above 100: 0
Negative protein values: 0


In [394]:
# Count missing values after cleaning
missing_after_cleaning = df_clean.isnull().sum()

print(missing_after_cleaning)

product_name          6904
categories_tags      97643
sugars_100g          52055
proteins_100g        38002
ingredients_text    130286
nutriscore_grade        12
dtype: int64


In [395]:
# Count product names that are empty or contain only spaces
empty_names = df_clean["product_name"].str.strip().eq("").sum()

print("Empty product names:", empty_names)

Empty product names: 2


In [396]:
# Replace empty product names with NaN
df_clean["product_name"] = df_clean["product_name"].str.strip()

df_clean["product_name"] = df_clean["product_name"].replace("", float("nan"))

# Count all missing product names
print("Missing product names:", df_clean["product_name"].isnull().sum())

Missing product names: 6906


In [397]:
# Remove rows with missing product names
df_clean = df_clean.dropna(subset=["product_name"])

In [398]:
# Check the new dataset shape
print("Dataset shape after removing missing names:", df_clean.shape)

Dataset shape after removing missing names: (493094, 6)


In [399]:
print("Missing product names:", df_clean["product_name"].isnull().sum())

Missing product names: 0


In [400]:
# Check missing sugar and protein values
print("Missing sugar values:", df_clean["sugars_100g"].isnull().sum())

print("Missing protein values:", df_clean["proteins_100g"].isnull().sum())

Missing sugar values: 46644
Missing protein values: 32699


In [401]:
# Keep products with both sugar and protein measurements
df_nutrition = df_clean.dropna(
    subset=["sugars_100g", "proteins_100g"]
)

# Check the new dataset shape
print("Nutrition dataset shape:", df_nutrition.shape)

Nutrition dataset shape: (445136, 6)


In [402]:
# Check the minimum and maximum sugar values
print("Minimum sugar:", df_nutrition["sugars_100g"].min())
print("Maximum sugar:", df_nutrition["sugars_100g"].max())

# Check the minimum and maximum protein values
print("Minimum protein:", df_nutrition["proteins_100g"].min())
print("Maximum protein:", df_nutrition["proteins_100g"].max())

Minimum sugar: 0.0
Maximum sugar: 100.0
Minimum protein: 0.0
Maximum protein: 100.0


In [403]:
# Export the cleaned dataset
df_clean.to_csv(
    "../data/processed/openfoodfacts_cleaned.csv",
    index=False
)

print("Cleaned dataset exported successfully!")

Cleaned dataset exported successfully!


In [404]:
# Display the first 20 category values
df_clean["categories_tags"].head(10)

0    en:breakfasts,en:spreads,en:sweet-spreads,fr:p...
1                                              en:null
2    en:plant-based-foods-and-beverages,en:beverage...
3    en:beverages-and-beverages-preparations,en:pla...
4                                                  NaN
5                                              en:null
6                                              en:null
7                                              en:null
8    en:plant-based-foods-and-beverages,en:beverage...
9                                              en:null
Name: categories_tags, dtype: object

In [405]:
# Check missing category values
missing_categories = df_clean["categories_tags"].isna().sum()

missing_percentage = (
    missing_categories / len(df_clean)
) * 100

print("Missing categories:", missing_categories)
print("Percentage missing:", round(missing_percentage, 2), "%")

Missing categories: 92201
Percentage missing: 18.7 %


In [406]:
# Find the 20 most common category combinations
df_clean["categories_tags"].value_counts().head(20)

categories_tags
en:undefined                                                                                                                     34361
en:snacks                                                                                                                        12505
en:snacks,en:sweet-snacks,en:confectioneries                                                                                      5417
en:dairies,en:fermented-foods,en:fermented-milk-products,en:cheeses                                                               5096
en:condiments,en:sauces,en:groceries                                                                                              4935
en:snacks,en:sweet-snacks,en:biscuits-and-cakes,en:biscuits                                                                       3955
en:desserts,en:frozen-foods,en:frozen-desserts                                                                                    3304
en:snacks,en:sweet-snacks,en:cocoa-and-

In [407]:
# Count products with undefined categories
df_clean["categories_tags"].str.contains(
    "en:undefined",
    na=False
).sum()

np.int64(34621)

In [408]:
# Display 10 products with undefined categories
df_clean.loc[
    df_clean["categories_tags"].str.contains(
        "en:undefined",
        na=False
    ),
    ["product_name", "categories_tags", "sugars_100g", "proteins_100g"]
].head(10)

,product_name,categories_tags,sugars_100g,proteins_100g
2341,Almonds,en:undefined,11.10,66.700000
2342,Trail Mix,en:undefined,88.90,44.400000
2343,Sesame Sticks,en:undefined,0.00,33.300000
2345,Double Chocolate Molten Upside Down Cake,en:undefined,24.50,4.900000
2346,Mini Cinnamon Loaf Cake,en:undefined,40.60,4.690000
2349,vitamin D milk,en:undefined,5.00,3.333333
2350,Large Topped Muffins,en:undefined,22.70,3.910000
2353,Reduce Fat Milk,en:undefined,4.80,3.200000
2361,Vitamin D Milk,en:undefined,5.00,3.333333
2362,Grade A Fat Free Milk,en:undefined,2.11,1.590000


In [409]:
# Display products where undefined appears with other categories
undefined_categories = df_clean.loc[
    df_clean["categories_tags"].str.contains(
        "en:undefined",
        na=False
    ),
    "categories_tags"
]

undefined_categories.value_counts().head(15)

categories_tags
en:undefined                                                                                                                                                                                                                                   34361
en:dairies,en:fermented-foods,en:fermented-milk-products,en:cheeses,en:undefined                                                                                                                                                                  20
en:condiments,en:sauces,en:salad-dressings,en:undefined                                                                                                                                                                                            8
en:plant-based-foods-and-beverages,en:plant-based-foods,en:fruits-and-vegetables-based-foods,en:vegetable-based-foods-and-beverages,en:canned-foods,en:vegetables-based-foods,en:canned-plant-based-foods,en:canned-vegetables,en:undefined        7
en:c

In [410]:
# Create a copy of the category column
df_clean["categories_clean"] = df_clean["categories_tags"]

# Remove en:undefined from category values
df_clean["categories_clean"] = (
    df_clean["categories_clean"]
    .str.replace("en:undefined", "", regex=False)
)

# Remove unnecessary commas at the beginning or end
df_clean["categories_clean"] = (
    df_clean["categories_clean"]
    .str.strip(",")
)

# Replace empty strings with missing values
df_clean["categories_clean"] = (
    df_clean["categories_clean"]
    .replace("", np.nan)
)

In [411]:
# Check the first 20 cleaned categories
df_clean["categories_clean"].head(20)

0     en:breakfasts,en:spreads,en:sweet-spreads,fr:p...
1                                               en:null
2     en:plant-based-foods-and-beverages,en:beverage...
3     en:beverages-and-beverages-preparations,en:pla...
4                                                   NaN
5                                               en:null
6                                               en:null
7                                               en:null
8     en:plant-based-foods-and-beverages,en:beverage...
9                                               en:null
10    en:plant-based-foods-and-beverages,en:beverage...
11    en:plant-based-foods-and-beverages,en:beverage...
12                                              en:null
13    en:plant-based-foods-and-beverages,en:plant-ba...
14                                           en:dairies
15                                              en:fats
16    en:plant-based-foods-and-beverages,en:plant-ba...
17    en:meats-and-their-products,en:meats,en:po

In [412]:
# Count products containing en:null
df_clean["categories_clean"].str.contains(
    "en:null",
    na=False
).sum()

np.int64(1026)

In [413]:
# Examine the most common category combinations containing en:null
df_clean.loc[
    df_clean["categories_clean"].str.contains(
        "en:null",
        na=False
    ),
    "categories_clean"
].value_counts().head(15)

categories_clean
en:null                                                                                                                                                                                                                                                                                                                1018
en:beverages-and-beverages-preparations,en:plant-based-foods-and-beverages,en:beverages,en:hot-beverages,en:plant-based-beverages,en:beverage-preparations,en:teas,en:tea-leaves,en:tea-bags,en:null                                                                                                                      1
en:condiments,en:vinegars,en:balsamic-vinegars,en:traditional-balsamic-vinegars,en:traditional-balsamic-vinegar-of-modena,en:null                                                                                                                                                                                         1
en:plant-based-foods-and-beverages,

In [414]:
# Remove en:null from category values
df_clean["categories_clean"] = (
    df_clean["categories_clean"]
    .str.replace("en:null", "", regex=False)
)

# Remove unnecessary commas
df_clean["categories_clean"] = (
    df_clean["categories_clean"]
    .str.strip(",")
)

# Replace empty strings with missing values
df_clean["categories_clean"] = (
    df_clean["categories_clean"]
    .replace("", np.nan)
)

In [415]:
# Check whether en:null still exists
df_clean["categories_clean"].str.contains(
    "en:null",
    na=False
).sum()

np.int64(0)

In [416]:
# Count missing values in the cleaned category column
df_clean["categories_clean"].isna().sum()

np.int64(127580)

In [417]:
# Calculate percentage of missing categories after cleaning
missing_percentage = (
    df_clean["categories_clean"].isna().sum()
    / len(df_clean)
) * 100

print(missing_percentage)

25.873362888211982


In [418]:
# Count products with available cleaned categories
df_clean["categories_clean"].notna().sum()

np.int64(365514)

In [419]:
# Select products with available categories
category_list = df_clean["categories_clean"].dropna()

# Split category combinations at commas
category_list = category_list.str.split(",")

# Display the first 10 results
category_list.head(10)

0     [en:breakfasts, en:spreads, en:sweet-spreads, ...
2     [en:plant-based-foods-and-beverages, en:bevera...
3     [en:beverages-and-beverages-preparations, en:p...
8     [en:plant-based-foods-and-beverages, en:bevera...
10    [en:plant-based-foods-and-beverages, en:bevera...
11    [en:plant-based-foods-and-beverages, en:bevera...
13    [en:plant-based-foods-and-beverages, en:plant-...
14                                         [en:dairies]
15                                            [en:fats]
16    [en:plant-based-foods-and-beverages, en:plant-...
Name: categories_clean, dtype: object

In [420]:
# Separate each category into its own row
category_list = category_list.explode()

# Display the first 20 categories
category_list.head(20)

0                              en:breakfasts
0                                 en:spreads
0                           en:sweet-spreads
0                        fr:pates-a-tartiner
0                        en:hazelnut-spreads
0                       en:chocolate-spreads
0             en:cocoa-and-hazelnuts-spreads
2         en:plant-based-foods-and-beverages
2                               en:beverages
2                           en:hot-beverages
2                   en:plant-based-beverages
2                                    en:teas
2                                en:tea-bags
3    en:beverages-and-beverages-preparations
3         en:plant-based-foods-and-beverages
3                               en:beverages
3                           en:hot-beverages
3                   en:plant-based-beverages
3                   en:beverage-preparations
3                                    en:teas
Name: categories_clean, dtype: object

In [421]:
#Count the product in each category
category_counts = category_list.value_counts()

category_counts.head(20)

categories_clean
en:plant-based-foods-and-beverages         111687
en:plant-based-foods                        94981
en:snacks                                   79334
en:sweet-snacks                             55737
en:beverages                                44387
en:dairies                                  37862
en:meats-and-their-products                 35061
en:cereals-and-potatoes                     33059
en:fruits-and-vegetables-based-foods        32779
en:condiments                               29507
en:fermented-foods                          29446
en:fermented-milk-products                  28959
en:meats                                    27952
en:beverages-and-beverages-preparations     27432
en:biscuits-and-cakes                       26770
en:meals                                    26512
en:desserts                                 26031
en:cereals-and-their-products               24768
en:groceries                                22608
en:frozen-foods                  

In [422]:
def assign_category(category):

    if pd.isna(category):
        return "Unclassified"

    # Split the category string into individual tags
    category_tags = category.lower().split(",")

    # Remove extra spaces
    category_tags = [tag.strip() for tag in category_tags]

    if any(tag.startswith(("en:biscuits", "en:cakes",
                          "en:pastries", "en:breads",
                          "en:bakery")) for tag in category_tags):
        return "Bakery"

    elif any(tag.startswith(("en:dairies", "en:dairy",
                             "en:milk", "en:yogurts",
                             "en:cheeses", "en:fermented-milk"))
             for tag in category_tags):
        return "Dairy"

    elif any(tag.startswith(("en:beverages", "en:beverage",
                             "en:drinks", "en:plant-based-beverages",
                             "en:hot-beverages"))
             for tag in category_tags):
        return "Beverages"

    elif any(tag.startswith(("en:snacks", "en:sweet-snacks",
                             "en:salty-snacks", "en:snack-bars",
                             "en:popcorn", "en:chips", "en:crisps"))
             or tag in ["en:nuts", "en:seeds"]
             for tag in category_tags):
        return "Snacks"

    elif any(tag.startswith(("en:fruits", "en:fruit",
                             "en:vegetables", "en:vegetable"))
             for tag in category_tags):
        return "Fruits & Vegetables"

    else:
        return "Other Foods"

# Apply the function to the categories_tags column
df_clean["Primary Category"] = df_clean["categories_tags"].apply(
    assign_category
)

# Check the classification results
df_clean["Primary Category"].value_counts()

#Check if the classifiation worked


Primary Category
Other Foods            182766
Unclassified            92201
Snacks                  57917
Dairy                   42570
Beverages               41666
Fruits & Vegetables     38529
Bakery                  37445
Name: count, dtype: int64

In [423]:
df_clean.loc[
    df_clean["Primary Category"] == "Other Foods",
    ["categories_clean", "Primary Category"]
].head(15)

,categories_clean,Primary Category
0,"en:breakfasts,en:spreads,en:sweet-spreads,fr:p...",Other Foods
1,NaN,Other Foods
5,NaN,Other Foods
6,NaN,Other Foods
7,NaN,Other Foods
9,NaN,Other Foods
12,NaN,Other Foods
15,en:fats,Other Foods
17,"en:meats-and-their-products,en:meats,en:poultr...",Other Foods
20,NaN,Other Foods


In [424]:
df_clean["Primary Category"].value_counts()

Primary Category
Other Foods            182766
Unclassified            92201
Snacks                  57917
Dairy                   42570
Beverages               41666
Fruits & Vegetables     38529
Bakery                  37445
Name: count, dtype: int64

In [425]:
df_clean["Primary Category"] = (
    df_clean["categories_clean"].apply(assign_category)
)

print("Primary Category" in df_clean.columns)
df_clean["Primary Category"].value_counts()

True


Primary Category
Other Foods            147387
Unclassified           127580
Snacks                  57917
Dairy                   42570
Beverages               41666
Fruits & Vegetables     38529
Bakery                  37445
Name: count, dtype: int64

In [426]:
#Add primary category to the nutrition dataset
df_nutrition = df_nutrition.copy()
df_nutrition["Primary Category"] = (
    df_clean["Primary Category"].loc[df_nutrition.index]
)
df_nutrition.head()

,product_name,categories_tags,sugars_100g,proteins_100g,ingredients_text,nutriscore_grade,Primary Category
0,Véritable pâte à tartiner noisettes chocolat noir,"en:breakfasts,en:spreads,en:sweet-spreads,fr:p...",32.000000,8.00000,"HAZELNUTS, 73% cocoa dark chocolate (cocoa mas...",e,Other Foods
14,Canola Harvest® Original Vegetable Oil Spread Tub,en:dairies,0.000000,0.00000,"CANOLA OIL, WATER, PALM OIL, PALM KERNEL OIL, ...",e,Dairy
15,"Canola harvest, buttery spread, with flaxseed oil",en:fats,0.000000,0.00000,"Canola oil, water, palm oil, flax oil, palm ke...",d,Other Foods
16,Lithuanian Rye Bread,"en:plant-based-foods-and-beverages,en:plant-ba...",15.385965,9.22807,"RYE FLOUR, WATER, WHEAT, FLOUR, MALT, MOLASSES...",e,Bakery
20,Magic Stars Chocolates,en:null,53.600000,6.40000,"SUGAR, COCOA BUTTER, SKIMMED MILK POWDER, COCO...",unknown,Unclassified


In [427]:
df_nutrition[["sugars_100g", "proteins_100g"]].describe()

,sugars_100g,proteins_100g
count,445136.000000,445136.000000
mean,14.716148,8.397291
std,20.793980,10.484646
min,0.000000,0.000000
25%,0.800000,1.000000
50%,4.310000,5.500000
75%,21.210000,12.000000
max,100.000000,100.000000


In [428]:
high_protein_low_sugar = df_nutrition[
    (df_nutrition["proteins_100g"] > 12) &
    (df_nutrition["sugars_100g"] < 4.31)
]

print("Products in opportunity quadrant:", len(high_protein_low_sugar))

Products in opportunity quadrant: 83328


In [429]:
opportunity_by_category = (
    high_protein_low_sugar["Primary Category"]
    .value_counts()
)

opportunity_by_category

Primary Category
Other Foods            42051
Unclassified           19079
Dairy                  14047
Snacks                  6190
Bakery                  1226
Fruits & Vegetables      448
Beverages                287
Name: count, dtype: int64

In [430]:
# Count all products in each category
total_by_category = df_nutrition["Primary Category"].value_counts()

# Count qualifying products in each category
opportunity_by_category = high_protein_low_sugar["Primary Category"].value_counts()

# Combine the counts
category_analysis = pd.DataFrame({
    "Total Products": total_by_category,
    "High Protein Low Sugar": opportunity_by_category
})

# Calculate the percentage of qualifying products
category_analysis["Opportunity Rate (%)"] = (
    category_analysis["High Protein Low Sugar"]
    / category_analysis["Total Products"]
) * 100

# Sort by opportunity rate
category_analysis.sort_values(
    by="Opportunity Rate (%)",
    ascending=False
)

,Total Products,High Protein Low Sugar,Opportunity Rate (%)
Primary Category,,,
Dairy,40490,14047,34.692517
Other Foods,135812,42051,30.962654
Unclassified,108645,19079,17.560863
Snacks,55836,6190,11.086038
Bakery,36281,1226,3.379179
Fruits & Vegetables,35010,448,1.279634
Beverages,33062,287,0.868066


In [431]:


fig = px.scatter(
    df_nutrition,
    x="sugars_100g",
    y="proteins_100g",
    color="Primary Category",
    title="Sugar vs Protein by Product Category",
    labels={
        "sugars_100g": "Sugar per 100g (g)",
        "proteins_100g": "Protein per 100g (g)"
    },
    opacity=0.4,
    render_mode="webgl"
)

# Add our opportunity thresholds
fig.add_vline(
    x=4.31,
    line_dash="dash",
    line_color="red"
)

fig.add_hline(
    y=12,
    line_dash="dash",
    line_color="green"
)

fig.update_layout(
    width=1000,
    height=650
)

from IPython.display import HTML, display

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [432]:
# Identify products that meet our nutritional criteria
df_nutrition["Opportunity"] = (
    (df_nutrition["sugars_100g"] < 4.31) &
    (df_nutrition["proteins_100g"] > 12)
)

# Summarise products by category
category_summary = df_nutrition.groupby("Primary Category").agg(
    Total_Products=("product_name", "count"),
    Opportunity_Products=("Opportunity", "sum")
)

# Calculate opportunity percentage
category_summary["Opportunity_Rate"] = (
    category_summary["Opportunity_Products"] /
    category_summary["Total_Products"] * 100
)

# Display results
category_summary.sort_values(
    "Opportunity_Rate",
    ascending=False
)

,Total_Products,Opportunity_Products,Opportunity_Rate
Primary Category,,,
Dairy,40490,14047,34.692517
Other Foods,135812,42051,30.962654
Unclassified,108645,19079,17.560863
Snacks,55836,6190,11.086038
Bakery,36281,1226,3.379179
Fruits & Vegetables,35010,448,1.279634
Beverages,33062,287,0.868066


Strategic Recommendation

The analysis shows that Dairy has the highest proportion of products meeting the nutritional criteria, at 34.69%. However, since the client's objective is to introduce a healthy-snacking product line, the Snacks category is the most relevant focus.

Out of 55,836 snack products analysed, 6,190 (11.09%) meet the selected nutritional criteria of more than 12g of protein and less than 4.31g of sugar per 100g.

Based on the data, the biggest market opportunity for the client's healthy-snacking line is in the Snacks category, specifically targeting products with more than 12g of protein and less than 4.31g of sugar per 100g.

This represents a potential nutritional product opportunity based on the available product data. It does not establish consumer demand or sales performance.

In [433]:
# Select products that meet our nutritional criteria
df_opportunity = df_nutrition[
    df_nutrition["Opportunity"] == True
].copy()

# Check the number of qualifying products
print("Number of opportunity products:", len(df_opportunity))

# Check missing ingredient information
print("Missing ingredient descriptions:",
      df_opportunity["ingredients_text"].isna().sum())

Number of opportunity products: 83328
Missing ingredient descriptions: 26311


In [434]:
# Keep products that have ingredient descriptions
df_ingredients = df_opportunity.dropna(
    subset=["ingredients_text"]
).copy()

# Display a few ingredient descriptions
df_ingredients["ingredients_text"].head(10)

25     BLACK CHICKPEA GRAM FLOUR, SALT, RAISING AGENT...
57     WHEAT FLOUR, SALT (4,8%), PALM OIL, ACIDITY RE...
177    SHRIMP SALT, SODIUM PHOSPHATE (TO RETAIN MOIST...
179    STONE GROUND ORGANIC YELLOW CORN, EXPELLER PRE...
184                                  Vital wheat gluten.
196    o, sin gluten, sin lactosa, sin azücares lient...
207    More information on www.paulandlinh.com Mékong...
224    Chaudin de porc, sel, poivre, conservateur : E...
234    Farine de _blé_, _morue_, pommes de terre, oeu...
237    Protéines de soja et de blé réhydratées, Eau, ...
Name: ingredients_text, dtype: object

In [435]:
# Remove products with missing ingredient descriptions
df_ingredients = df_opportunity.dropna(
    subset=["ingredients_text"]
).copy()

# Count products containing different protein sources
protein_sources = {
    "Soy": "soy|soya",
    "Peanuts": "peanut",
    "Whey": "whey",
    "Milk": "milk",
    "Eggs": "egg",
    "Wheat/Gluten": "wheat|gluten",
    "Chickpeas": "chickpea",
    "Shrimp": "shrimp"
}

for source, keyword in protein_sources.items():
    count = df_ingredients["ingredients_text"].str.contains(
        keyword,
        case=False,
        na=False
    ).sum()

    print(source, ":", count)

Soy : 3860
Peanuts : 1416
Whey : 1224
Milk : 8090
Eggs : 1242
Wheat/Gluten : 7246
Chickpeas : 124
Shrimp : 589


Bonus Analysis: Common Protein Sources

The ingredient analysis examined 57,017 qualifying products with available ingredient descriptions.

The three most frequently identified protein-related ingredient sources were:

- Milk: 8,090 products
- Wheat/Gluten: 7,246 products
- Soy: 3,860 products

This suggests that milk-based, wheat/gluten-based and soy-based ingredients are common among products meeting the selected nutritional criteria.

The analysis uses keyword matching in ingredient descriptions. Therefore, the counts represent product mentions and do not measure the actual protein contribution of each ingredient.

In [436]:
import plotly.express as px

# Sort categories by number of opportunity products
category_counts = category_summary.sort_values(
    "Opportunity_Products",
    ascending=False
).reset_index()

# Create bar chart
fig = px.bar(
    category_counts,
    x="Primary Category",
    y="Opportunity_Products",
    title="Number of Products Meeting Nutritional Opportunity Criteria",
    labels={
        "Primary Category": "Product Category",
        "Opportunity_Products": "Number of Products"
    },
    text="Opportunity_Products"
)

fig.update_layout(
    xaxis_title="Product Category",
    yaxis_title="Number of Opportunity Products"
)

from IPython.display import HTML, display

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

 Candidate's Choice: Opportunity Products by Category

This chart compares the number of products meeting the nutritional opportunity criteria across the seven product categories.

It was selected to complement the opportunity-rate analysis by showing the actual number of qualifying products in each category. This helps identify categories with larger groups of products that could inform further product development research.

In [437]:
# Select the columns needed for the dashboard
dashboard_data = df_nutrition[
    [
        "product_name",
        "Primary Category",
        "sugars_100g",
        "proteins_100g",
        "ingredients_text",
        "Opportunity"
    ]
].copy()

# Export the dashboard dataset
dashboard_data.to_csv(
    "../data/processed/dashboard_data.csv",
    index=False
)

print("Dashboard dataset saved successfully!")
print("Dataset shape:", dashboard_data.shape)

Dashboard dataset saved successfully!
Dataset shape: (445136, 6)


Dashboard Data Preparation

In [438]:
dashboard_data = dashboard_data[
    [
        "product_name",
        "Primary Category",
        "sugars_100g",
        "proteins_100g",
        "Opportunity"
    ]
]

dashboard_data.to_csv(
    "../data/processed/dashboard_data.csv",
    index=False
)